# safety_runs

> Phase 9 runs: a configuration with bad agents and a defence, the team builder, one run, its flat row (honest and team welfare, monitoring cost, sanctions, detection AUC) and the pairing of attacked with clean runs.

Spec: `specs/multiagent/phase9/` (design-low-level section 4).

In [ ]:
#| default_exp safety_runs

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass

import numpy as np

from explRL.adversary import BAD
from explRL.bench_metrics import discoveries, freeRiding
from explRL.defence import DefendedBandit, Monitor
from explRL.harness import Blackboard, MessageBus
from explRL.harness_agents import LeadFollower
from explRL.harness_runs import HarnessConfig, benchWorld, makeChannel, netHarvest, worldTime
from explRL.multi import runTeam, spawnRngs
from explRL.team_refs import ledgerArrays

NAN = float('nan')


@dataclass
class SafetyConfig(HarnessConfig):
    kind: str = 'none'          # 'none' | 'liar' | 'freerider' | 'saboteur'
    nBad: int = 0
    defence: str = 'none'       # 'none' | 'trust' | 'reputation' | 'sanctions'
    q: float = 0.0
    theta: float = 2.5          # T7 calibration


def makeTeam9(cfg, N, seed, channel, monitor):
    "Bad agents of `cfg.kind` in slots 0..nBad-1, honest DefendedBandits (or LeadFollowers) elsewhere."
    if cfg.nBad and cfg.kind not in BAD:
        raise ValueError(f'unknown bad-agent kind {cfg.kind}')
    if (cfg.defence != 'none' or cfg.q > 0) and not isinstance(channel, (Blackboard, MessageBus)):
        raise ValueError('defences need the blackboard or messages channel')
    rngs, team = spawnRngs(seed + 3000, N), []
    for i in range(N):
        if cfg.channel == 'hierarchical':
            team.append(BAD[cfg.kind][1](channel) if i < cfg.nBad else LeadFollower(channel))
        elif i < cfg.nBad:
            team.append(BAD[cfg.kind][0](channel, cfg.w, cfg.K, cfg.cUCB, cfg.H))
        else:
            team.append(DefendedBandit(channel, cfg.defence, monitor, cfg.q, rngs[i], w=cfg.w, K=cfg.K, cUCB=cfg.cUCB, H=cfg.H))
    return team


def auc(scores, isBad):
    "P(score of a random bad agent > score of a random honest agent), ties 1/2; nan scores dropped."
    s, b = np.asarray(scores, float), np.asarray(isBad, bool)
    pos, neg = s[b & ~np.isnan(s)], s[~b & ~np.isnan(s)]
    if not len(pos) or not len(neg):
        return NAN
    d = pos[:, None] - neg[None, :]
    return float(((d > 0) + 0.5 * (d == 0)).mean())


def _share(x):
    return float(np.mean(x)) if len(x) else NAN


def safetyRow(cfg, world, hist, channel, monitor, members, N, T, seed):
    "One flat row: honest and team welfare, monitoring, sanctions, detection, discoveries."
    L, b = ledgerArrays(world.ledger), cfg.nBad
    net, isBad = netHarvest(world, L), np.arange(N) < b
    row = dict(seed=seed, N=N, Trun=T, channel=cfg.channel, budget=cfg.budget, kind=cfg.kind, nBad=b,
               defence=cfg.defence, q=cfg.q, theta=cfg.theta, events=len(hist),
               Wh=float(net[L['agent'] >= b].sum()) / ((N - b) * T), Wt=float(net.sum()) / (N * T))
    dm = [m for m in members if isinstance(m, DefendedBandit)]
    row.update(monitorCost=sum(m.checkSeconds for m in dm) / (N * T), checks=sum(m.checks for m in dm),
               deliberate=sum(m.deliberate for m in dm))
    lv = monitor.maxLevel if monitor is not None and monitor.sanctions else None
    rev = monitor.firstRevoke[:b] if lv is not None else np.array([])
    row.update(badLevel2=_share(lv[:b] >= 2) if lv is not None else NAN, badRevoked=_share(lv[:b] >= 3) if lv is not None else NAN,
               honestLevel2=_share(lv[b:] >= 2) if lv is not None else NAN,
               honestRevoked=_share(lv[b:] >= 3) if lv is not None else NAN,
               firstRevoke=float(np.nanmin(rev)) if np.isfinite(rev).any() else NAN)
    peer = isinstance(channel, (Blackboard, MessageBus))
    white = [m.whiteScore() if hasattr(m, 'whiteScore') else NAN for m in members]
    if cfg.defence == 'trust':
        given = np.array([_share([m.tau[s] for m in dm if m.id != s]) for s in range(N)])
    elif monitor is not None:
        given = monitor.rho.copy()
    else:
        given = np.full(N, NAN)
    liar = cfg.kind == 'liar' and b > 0 and peer
    row.update(aucWhite=auc(white, isBad) if liar else NAN,
               aucBlack=auc(1 - given, isBad) if liar and cfg.defence != 'none' else NAN,
               trustBad=_share(given[:b]) if b and cfg.defence != 'none' else NAN,
               trustHonest=_share(given[b:]) if cfg.defence != 'none' else NAN)
    disc = discoveries(world, L)
    row.update(discoveries=int(disc.sum()), freeRiding=freeRiding(L, disc, N))
    return row


def runSafety(cfg, N, seed):
    "Build the benchmark world, channel, monitor and team (bad slots first), run it; returns world, hist, channel, row."
    world = benchWorld(**cfg.world)
    world.reset(seed, N)
    channel = makeChannel(cfg, world.L, N, seed)
    monitor = Monitor(N, cfg.defence == 'sanctions', cfg.theta, channel=channel) \
        if cfg.defence in ('reputation', 'sanctions') else None
    members = makeTeam9(cfg, N, seed, channel, monitor)
    T = worldTime(N, cfg)
    hist = runTeam(members, world, T)
    return world, hist, channel, safetyRow(cfg, world, hist, channel, monitor, members, N, T, seed)


def lossRows(rows):
    "Pair each attacked row with its clean row (same study, channel, defence, q, N, seed): relative honest and team loss."
    key = lambda r: (r['study'], r['channel'], r['defence'], float(r['q']), int(r['N']), int(r['seed']))
    clean = {key(r): r for r in rows if int(r['nBad']) == 0}
    out = []
    for r in rows:
        c = clean.get(key(r))
        if int(r['nBad']) > 0 and c is not None:
            out.append(dict(study=r['study'], factor=r.get('factor', ''), channel=r['channel'], budget=r['budget'],
                            defence=r['defence'], q=r['q'], kind=r['kind'], nBad=r['nBad'], N=r['N'], seed=r['seed'],
                            loss=1 - r['Wh'] / c['Wh'] if c['Wh'] > 0 else NAN,
                            lossTeam=1 - r['Wt'] / c['Wt'] if c['Wt'] > 0 else NAN))
    return out

## Tests

In [ ]:
import time
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.safety_runs import SafetyConfig, makeTeam9, auc, safetyRow, runSafety, lossRows
from explRL.defence import DefendedBandit
from explRL.harness_runs import HarnessConfig, benchWorld, runHarness

# AC-1 bridge: no bad agents, no defence, q = 0 gives runHarness's history (every channel)
for ch in ('none', 'blackboard', 'messages', 'hierarchical'):
    for seed in (0, 1, 2):
        a = runSafety(SafetyConfig(channel=ch, T=200.0), 8, seed)[1].records
        test_eq(a, runHarness(HarnessConfig(channel=ch, T=200.0), 8, seed)[1].records)
d = SafetyConfig(channel='messages', kind='liar', nBad=2, defence='trust', q=0.05).asDict()
test_eq(SafetyConfig.fromDict(d).asDict(), d)

# AC-5: honest and team welfare on a hand-built ledger (agent 0 bad: harvest 5; agents 1, 2 harvest 1 and 2; 10 s)
w = benchWorld(nPatches=1, nDanger=0, intake=(5.0,), patchSites=[0], dangerSites=[]); w.reset(0, 3)
w.ledger = [(i, 0.0, 1.0, 'stay', 0, 0, h, h, 0.0, 1.0) for i, h in ((0, 5.0), (1, 1.0), (2, 2.0))]
r = safetyRow(SafetyConfig(kind='liar', nBad=1), w, [], None, None, [DefendedBandit(None) for _ in range(3)], 3, 10.0, 0)
test_close((r['Wh'], r['Wt']), (3.0 / 20, 8.0 / 30))
test_eq((auc([2, 3, 0, 1], [1, 1, 0, 0]), auc([1, 1], [1, 0]), auc([0, 1], [1, 0])), (1.0, 0.5, 0.0))
db = DefendedBandit(None); db.reportLog = [(0, 0.5, 0.5, 0.0)]; test_eq(db.whiteScore(), 0.0)
rows = [dict(study='h3', channel='blackboard', defence='none', q=0.0, N=8, seed=1, nBad=n, kind=k, budget='compute',
             Wh=wh, Wt=wt) for n, k, wh, wt in ((0, 'none', 0.5, 0.5), (1, 'liar', 0.4, 0.45))]
lr = lossRows(rows + [dict(rows[1], seed=2)])                                       # an unpaired row is skipped
test_eq(len(lr), 1); test_close((lr[0]['loss'], lr[0]['lossTeam']), (0.2, 0.1))

# attacks run end to end; liars are told apart from honest agents by the white-box score and by sanctions
cfg = SafetyConfig(channel='blackboard', kind='liar', nBad=4, defence='sanctions', q=0.05, T=300.0)
w, h, ch, row = runSafety(cfg, 16, 1000)
assert row['aucWhite'] > 0.9 and row['checks'] > 0 and row['monitorCost'] > 0, row
for kind in ('freerider', 'saboteur'):
    for chn in ('none', 'messages', 'hierarchical'):
        assert runSafety(SafetyConfig(channel=chn, kind=kind, nBad=2, T=60.0), 8, 0)[3]['Wh'] == \
               runSafety(SafetyConfig(channel=chn, kind=kind, nBad=2, T=60.0), 8, 0)[3]['Wh']
try:
    makeTeam9(SafetyConfig(channel='hierarchical', defence='trust'), 4, 0, None, None)
    raise AssertionError('expected ValueError')
except ValueError:
    pass

# speed: a 64-agent blackboard team with sanctions and spot checks at fixed compute
t0 = time.perf_counter()
_, hist, _, row = runSafety(SafetyConfig(channel='blackboard', kind='liar', nBad=16, defence='sanctions', q=0.05,
                                         budget='compute'), 64, 1000)
el = time.perf_counter() - t0
print(f'64 agents, sanctions, q = 0.05: {el:.2f} s, {len(hist)} events, {el / len(hist) * 1e6:.0f} us/event')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()